# Time-Series — โจทย์แบบ "ทำนายค่าในอนาคต" (Forecasting)

มีค่าตามเวลาในอดีต -> ทำนายค่าในอนาคต (เช่น ยอดขายพรุ่งนี้, ราคาสัปดาห์หน้า)

วิธีในโน้ตบุ๊กนี้ (มือใหม่):
- วิธีที่ 1 (ง่ายสุด) = สร้างฟีเจอร์จากวันที่ + ค่าย้อนหลัง (lag) แล้วใช้ `LightGBM` ทำนายเหมือน regression
- วิธีที่ 2 (ไม่บังคับ) = `AutoGluon TimeSeries` (เฉพาะทาง forecasting)


## เมื่อไหร่ใช้โน้ตบุ๊กนี้

ใช้เมื่อต้องทำนาย "ค่าตัวเลขในอนาคต" จากประวัติตามเวลา (มีคอลัมน์วันที่/เวลา)
ถ้าทำนาย "คลาสจากหน้าต่างสัญญาณ" -> ไปใช้ `signal_classification.ipynb`

ต้องแก้ (`# << แก้`): ชื่อ competition, `TIME_COL` (คอลัมน์วันที่), `TARGET`, `SERIES_COL` (ถ้ามีหลายซีรีส์)

## ขั้นที่ 1 — ติดตั้ง

In [ ]:
!pip -q install lightgbm scikit-learn pandas numpy
!pip -q install -U "autogluon.timeseries"   # วิธีที่ 2 เท่านั้น

## ขั้นที่ 2 — ตั้งค่า Kaggle แล้วดาวน์โหลดข้อมูล

ต้องมีไฟล์ `kaggle.json` ก่อน วิธีได้มา: เข้า kaggle.com -> กดรูปโปรไฟล์ -> Settings -> Account -> Create New Token
จะได้ไฟล์ `kaggle.json` หน้าตา `{"username":"...","key":"..."}`

- ถ้ารันบน `Kaggle Notebook`: ข้อมูลอยู่ที่ `/kaggle/input/...` แล้ว ไม่ต้องทำอะไร รันผ่านได้เลย
- ถ้ารันบน `Google Colab / เครื่องตัวเอง`: เอา username กับ key ใส่ในเซลล์ข้างล่าง (จุด `# << แก้`)

In [ ]:
import os, glob, subprocess

COMP = "ใส่-slug-ของ-competition-forecasting-ตรงนี้"      # << แก้ตรงนี้: slug ของ competition (คือส่วนท้าย URL หลังคำว่า /competitions/)
KAGGLE_USERNAME = ""   # << แก้ (เฉพาะ Colab/เครื่องตัวเอง) เช่น "peetwan1"
KAGGLE_KEY      = ""   # << แก้ (เฉพาะ Colab/เครื่องตัวเอง) คีย์ยาว ๆ จาก kaggle.json

def get_data(comp):
    # ถ้าอยู่บน Kaggle ข้อมูลถูกต่อไว้ให้แล้ว
    kpath = f"/kaggle/input/{comp}"
    if os.path.isdir(kpath):
        print("อยู่บน Kaggle แล้ว ข้อมูลอยู่ที่", kpath); return kpath
    # ถ้าไม่ใช่ Kaggle: เขียน token แล้วโหลดด้วย kaggle CLI
    if KAGGLE_USERNAME and KAGGLE_KEY:
        os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
        open(os.path.expanduser("~/.kaggle/kaggle.json"), "w").write(
            '{"username":"%s","key":"%s"}' % (KAGGLE_USERNAME, KAGGLE_KEY))
        os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)
    os.makedirs("data", exist_ok=True)
    subprocess.run(["kaggle","competitions","download","-c",comp,"-p","data"], check=True)
    for z in glob.glob("data/*.zip"):
        subprocess.run(["unzip","-o","-q",z,"-d","data"], check=True)
    return "data"

DATA_DIR = get_data(COMP)
print("ไฟล์ที่ดาวน์โหลดมา (ดูชื่อไฟล์/โฟลเดอร์ แล้วเอาไปแก้ในเซลล์ถัดไป):")
for p in sorted(glob.glob(os.path.join(DATA_DIR, "**"), recursive=True))[:40]:
    print("  ", p)

## ขั้นที่ 3 — โหลดข้อมูล + CONFIG

In [ ]:
import os, pandas as pd, numpy as np
TRAIN_CSV=os.path.join(DATA_DIR,"train.csv"); TEST_CSV=os.path.join(DATA_DIR,"test.csv")
SAMPLE_SUB=os.path.join(DATA_DIR,"sample_submission.csv")
train=pd.read_csv(TRAIN_CSV); test=pd.read_csv(TEST_CSV); sample=pd.read_csv(SAMPLE_SUB)
TIME_COL  = "date"      # << แก้: คอลัมน์วันที่/เวลา
TARGET    = sample.columns[1]   # << แก้ถ้าผิด: คอลัมน์ค่าที่ทำนาย
SERIES_COL= None        # << แก้: ถ้ามีหลายซีรีส์ (เช่น "store_id") ใส่ชื่อ, ถ้าซีรีส์เดียวใส่ None
ID_COL=sample.columns[0]
print("คอลัมน์:",list(train.columns)); display(train.head()); display(sample.head())

## วิธีที่ 1 — ฟีเจอร์วันที่ + lag + LightGBM (ง่ายสุด ใช้ได้จริง)

หลักคิด: เปลี่ยนปัญหา forecasting ให้เป็น regression ธรรมดา โดยทำฟีเจอร์จากวันที่ (ปี/เดือน/วัน/วันในสัปดาห์)
+ ค่าย้อนหลัง (lag) แล้วให้ LightGBM เรียนรู้

In [ ]:
import lightgbm as lgb
def add_time_features(df):
    df=df.copy(); df[TIME_COL]=pd.to_datetime(df[TIME_COL])
    df["year"]=df[TIME_COL].dt.year; df["month"]=df[TIME_COL].dt.month
    df["day"]=df[TIME_COL].dt.day; df["dow"]=df[TIME_COL].dt.dayofweek
    df["doy"]=df[TIME_COL].dt.dayofyear
    return df
def add_lags(df, lags=(1,7,14,28)):   # << แก้ค่า lag ตามความถี่ข้อมูล (รายวัน/รายชั่วโมง)
    df=df.sort_values(TIME_COL).copy()
    g=df.groupby(SERIES_COL)[TARGET] if SERIES_COL else df[TARGET]
    for L in lags:
        df[f"lag_{L}"]= (g.shift(L) if SERIES_COL else df[TARGET].shift(L))
    return df
tr=add_lags(add_time_features(train)).dropna()
feat_cols=[c for c in tr.columns if c not in [TARGET,TIME_COL,ID_COL] and pd.api.types.is_numeric_dtype(tr[c])]
m=lgb.LGBMRegressor(n_estimators=2000,learning_rate=0.02,num_leaves=63,random_state=42,verbose=-1)
m.fit(tr[feat_cols], tr[TARGET])
# เตรียม test: ต้องมีฟีเจอร์เดียวกัน (ถ้า test ไม่มี lag ต้องต่อ history จาก train ก่อน) -- ปรับตามรูปแบบจริง
te=add_time_features(test)
for col in feat_cols:
    if col not in te.columns: te[col]=np.nan   # << แก้: เติม lag ของ test จาก history จริงถ้าจำเป็น
out=sample.copy(); out[TARGET]=m.predict(te[feat_cols])
out.to_csv("submission.csv",index=False); print("บันทึก submission.csv"); display(out.head())

## วิธีที่ 2 — AutoGluon TimeSeries (ไม่บังคับ เฉพาะทาง forecasting)

เหมาะเมื่อข้อมูลเป็นรูปแบบ long (item_id, timestamp, target) ชัดเจน

In [ ]:
# from autogluon.timeseries import TimeSeriesDataFrame, TimeSeriesPredictor
# tsdf = TimeSeriesDataFrame.from_data_frame(train, id_column=SERIES_COL, timestamp_column=TIME_COL)
# predictor = TimeSeriesPredictor(target=TARGET, prediction_length=28).fit(tsdf, time_limit=600)
# pred = predictor.predict(tsdf)   # แล้วแมปกลับเข้า sample_submission ตามรูปแบบจริง
print("ปลดคอมเมนต์เซลล์นี้ถ้าข้อมูลเป็นรูปแบบ long ชัดเจน")

## ขั้นสุดท้าย — ส่ง submission ขึ้น Kaggle

เช็คก่อนส่งทุกครั้ง: ไฟล์ submission ต้องมีชื่อคอลัมน์ + จำนวนแถว ตรงกับ `sample_submission.csv` เป๊ะ ๆ
- บน Kaggle Notebook: กดปุ่ม `Submit` ที่แถบขวา (ง่ายสุด) หรือใช้คำสั่งข้างล่าง
- บน Colab/เครื่อง: รันเซลล์ข้างล่าง (ต้องตั้ง token แล้ว)

In [ ]:
FILE = "submission.csv"   # << แก้เป็นชื่อไฟล์ที่คะแนนดีสุด
!kaggle competitions submit -c {COMP} -f {FILE} -m "lightgbm forecasting"
!kaggle competitions submissions -c {COMP} | head